In [3]:
import xarray as xr

Load checkpoint using their implementation 

In [1]:
from config import default_experiment_config
exp = default_experiment_config()

In [4]:
from forecast import generate_model

(model_config, task_config, params) = generate_model.load_model(exp.model.checkpoint_path)

diffs_stddev_by_level = xr.open_dataset(exp.model.stats_path + "diffs_stddev_by_level.nc")
mean_by_level = xr.open_dataset(exp.model.stats_path + "mean_by_level.nc")
stddev_by_level = xr.open_dataset(exp.model.stats_path + "stddev_by_level.nc")
predictor = generate_model.build_predictor(
    exp.compute, model_config, task_config, params, diffs_stddev_by_level=diffs_stddev_by_level, mean_by_level=mean_by_level, stddev_by_level=stddev_by_level
)

Data loading, from own source

In [8]:
def extend_time_with_nans(data: xr.Dataset, p: int) -> xr.Dataset:
    import numpy as np
    if "time" not in data.dims:
        raise ValueError("Input dataset must have a 'time' dimension.")

    n = data.sizes["time"]
    if n > p:
        raise ValueError(
            f"Dataset has {n} time steps, which exceeds requested length {p}."
        )

    # Generate new time coordinate of length p (with regular spacing inferred)
    time_vals = data["time"].values
    if n >= 2:
        time_step = time_vals[1] - time_vals[0]
    else:
        # fallback to arbitrary 6-hour step if only one timestep exists
        time_step = np.timedelta64(6, "h")

    new_time = np.array([time_vals[0] + i * time_step for i in range(p)])

    # Create empty dataset with same structure but time length p
    skeleton_vars = {}
    for var in data.data_vars:
        dims = data[var].dims
        shape = [p if d == "time" else data.sizes[d] for d in dims]
        skeleton_vars[var] = (dims, np.full(shape, np.nan, dtype=data[var].dtype))

    # Rebuild coordinates
    new_coords = {
        name: (dims, data.coords[name].values)
        for name, dims in data.coords.items()
        if "time" not in dims
    }
    new_coords["time"] = new_time

    # Create skeleton dataset
    padded = xr.Dataset(skeleton_vars, coords=new_coords)

    # Copy original data into the first `n` time steps
    padded.loc[dict(time=data["time"])] = data

    return padded

In [9]:
from graphcast.data_utils import extract_inputs_targets_forcings
import dataclasses
input_path = "/home/users/f/froelicm/scratch/Data/GraphCast_OP/custom-hres_2022-09-26_res-0.25_levels-13_steps-16.nc"
data = xr.load_dataset(input_path, engine="netcdf4").isel(time=slice(0,2))#.compute()
data = data.drop_vars('datetime')

/tmp/ipykernel_2264975/4293284012.py:4: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  data = xr.load_dataset(input_path, engine="netcdf4").isel(time=slice(0,2))#.compute()


In [10]:
data = extend_time_with_nans(data, 40)
leadtime = 40
inputs, targets, forcings = extract_inputs_targets_forcings(
    data,
    target_lead_times=slice("6h", f"{leadtime*6}h"),
    **dataclasses.asdict(task_config),
)

: 

In [5]:
# first time running this takes ~2min, then about 7sec for 14step forecast
predictions = predictor(inputs, targets, forcings)

ValueError: 'grid2mesh_gnn/~_networks_builder/encoder_nodes_grid_nodes_mlp/~/linear_0/w' with retrieved shape (184, 512) does not match shape=[10, 512] dtype=dtype(bfloat16)

In [17]:
import os
save_dir = "/home/users/f/froelicm/scratch/GraphCast-OP_TC_5day/AMSE"
save_name = "regional_ep-0.nc"
predictions.sel(
    lat=slice(5, 45),
    lon=slice(-120 + 360, -50 + 360),
    level=[1000, 850, 700, 500],
).to_netcdf(os.path.join(save_dir, save_name), engine="netcdf4")
del predictions